# Geocoordinates Map (Plotly)

This notebook visualizes store locations from `Geocoordinates.xlsx` on a **real map tile**. The **first row** (Mahboula Complex - Mix) is treated as the fixed **start/end depot** as referenced in `Approaches/approach.md`.


In [ ]:
import pandas as pd
import numpy as np

try:
    import plotly.graph_objects as go
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
    import plotly.graph_objects as go


In [ ]:
from pathlib import Path

data_path = Path("../datasets/Geocoordinates.xlsx")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path.resolve()}")

df = pd.read_excel(data_path)

# Normalize column names
rename_map = {
    "Store Name": "store_name",
    "Store ID": "store_id",
}

df = df.rename(columns=rename_map)

# Ensure numeric coordinates
for col in ["latitude", "longitude"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows with missing coordinates
before = len(df)
df = df.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
print(f"Loaded {len(df)} rows (dropped {before - len(df)} rows with missing coordinates)")

df.head()


In [ ]:
# The first row is the fixed start/end depot
if df.empty:
    raise ValueError("Dataset is empty after cleaning.")

depot = df.loc[0]
print("Depot (fixed start/end):", depot["store_name"], "| ID:", depot["store_id"])


In [ ]:
# Plot on real map tiles (OpenStreetMap)
center = {
    "lat": float(df["latitude"].mean()),
    "lon": float(df["longitude"].mean())
}

# Depot trace
fig = go.Figure()
fig.add_trace(go.Scattermapbox(
    lat=[depot["latitude"]],
    lon=[depot["longitude"]],
    mode="markers+text",
    marker=dict(size=14, color="red"),
    text=["START/END"],
    textposition="top right",
    hovertext=[f"START/END (Depot): {depot['store_name']} ({depot['store_id']})"],
    hoverinfo="text",
    name="Depot"
))

# Store points
others = df.iloc[1:].copy()
fig.add_trace(go.Scattermapbox(
    lat=others["latitude"],
    lon=others["longitude"],
    mode="markers",
    marker=dict(size=7, color="#1f77b4", opacity=0.8),
    hovertext=others.apply(lambda r: f"{r['store_name']} ({r['store_id']})", axis=1),
    hoverinfo="text",
    name="Stores"
))

fig.update_layout(
    mapbox_style="open-street-map",
    mapbox_center=center,
    mapbox_zoom=10,
    margin=dict(l=0, r=0, t=0, b=0),
    legend=dict(orientation="h", yanchor="bottom", y=0.01, xanchor="left", x=0.01)
)

fig
